# GraphRAG with Neo4j + Milvus local and FlatRAG Comparison

Notebook này xây dựng end-to-end một pipeline retrieval có 2 chế độ:

- `FlatRAG`: baseline, chỉ lấy 1 chunk liên quan nhất từ Milvus và không dùng graph.
- `GraphRAG`: dùng semantic search trên Milvus + Neo4j để lưu knowledge graph, mở rộng multi-hop, và gom bằng chứng từ graph.

Dữ liệu đầu vào hiện là duy nhất 1 file markdown: `data/wiki100k_samples/01_12.md` được tách từ `data/wiki100k.parquet`. Mọi tham số đều được gom vào config ở đầu notebook.

In [1]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '--upgrade',
    'python-dotenv',
    'pydantic',
    'neo4j',
    'pymilvus',
    'requests',
    'openai',
    'langchain-core',
    'langchain-openai',
    'pandas',
    'numpy',
    'scikit-learn',
    'tqdm',
    'matplotlib',
])

0

In [2]:
import os
import re
import json
import math
import hashlib
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from neo4j import GraphDatabase
from pymilvus import MilvusClient
from pydantic import BaseModel, Field
from tqdm.auto import tqdm

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

pd.set_option('display.max_colwidth', 200)

/opt/anaconda3/envs/lab19/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load environment variables from .env if available.
load_dotenv()


def env_list(name: str, default: str = '') -> List[str]:
    raw = os.getenv(name, default).strip()
    if not raw:
        return []
    return [item.strip() for item in raw.split(',') if item.strip()]


@dataclass
class Neo4jConfig:
    uri: str = os.getenv('NEO4J_URI', '')
    user: str = os.getenv('NEO4J_USERNAME', os.getenv('NEO4J_USER', ''))
    password: str = os.getenv('NEO4J_PASSWORD', '')
    database: str = os.getenv('NEO4J_DATABASE', 'neo4j')
    fulltext_index_name: str = os.getenv('NEO4J_FULLTEXT_INDEX_NAME', 'entity_fulltext')


@dataclass
class MilvusConfig:
    uri: str = os.getenv('MILVUS_URI', 'http://localhost:19530')
    token: str = os.getenv('MILVUS_TOKEN', '')
    collection_name: str = os.getenv('MILVUS_COLLECTION_NAME', 'graph_rag_chunks')
    metric_type: str = os.getenv('MILVUS_METRIC_TYPE', 'COSINE')
    drop_existing_collection: bool = os.getenv('MILVUS_DROP_EXISTING_COLLECTION', 'true').lower() == 'true'


@dataclass
class JinaConfig:
    api_key: str = os.getenv('JINA_API_KEY', '')
    embedding_model: str = os.getenv('JINA_EMBEDDING_MODEL', 'jina-embeddings-v3')
    api_base_url: str = os.getenv('JINA_API_BASE_URL', 'https://api.jina.ai/v1/embeddings')
    normalized: bool = os.getenv('JINA_NORMALIZED', 'true').lower() == 'true'
    embedding_type: str = os.getenv('JINA_EMBEDDING_TYPE', 'float')
    truncate: bool = os.getenv('JINA_TRUNCATE', 'true').lower() == 'true'


@dataclass
class ModelConfig:
    chat_model: str = os.getenv('OPENAI_CHAT_MODEL', '')
    temperature: float = float(os.getenv('LLM_TEMPERATURE', '0.0'))


@dataclass
class DataConfig:
    document_paths: List[str] = field(default_factory=lambda: env_list('DOCUMENT_PATHS'))
    chunk_size: int = int(os.getenv('CHUNK_SIZE', '1400'))
    chunk_overlap: int = int(os.getenv('CHUNK_OVERLAP', '180'))
    max_documents: Optional[int] = int(os.getenv('MAX_DOCUMENTS')) if os.getenv('MAX_DOCUMENTS') else None


@dataclass
class BuildConfig:
    max_entities_per_chunk: int = int(os.getenv('MAX_ENTITIES_PER_CHUNK', '8'))
    max_relationships_per_chunk: int = int(os.getenv('MAX_RELATIONSHIPS_PER_CHUNK', '10'))
    extraction_cache_path: str = os.getenv('EXTRACTION_CACHE_PATH', '.cache_graph_extraction.json')
    build_manifest_path: str = os.getenv('BUILD_MANIFEST_PATH', '.cache_build_manifest.json')
    skip_existing_build: bool = os.getenv('SKIP_EXISTING_BUILD', 'true').lower() == 'true'
    rebuild_neo4j: bool = os.getenv('REBUILD_NEO4J', 'true').lower() == 'true'
    truncate_existing_graph: bool = os.getenv('TRUNCATE_EXISTING_GRAPH', 'false').lower() == 'true'


@dataclass
class RetrievalConfig:
    flat_top_k: int = int(os.getenv('FLAT_TOP_K', '1'))
    graph_seed_k: int = int(os.getenv('GRAPH_SEED_K', '5'))
    graph_hops: int = int(os.getenv('GRAPH_HOPS', '2'))
    graph_neighbor_limit_per_node: int = int(os.getenv('GRAPH_NEIGHBOR_LIMIT_PER_NODE', '20'))
    graph_chunk_boost_k: int = int(os.getenv('GRAPH_CHUNK_BOOST_K', '5'))
    max_context_chunks: int = int(os.getenv('MAX_CONTEXT_CHUNKS', '8'))
    max_context_chars: int = int(os.getenv('MAX_CONTEXT_CHARS', '12000'))
    flat_max_context_chars: int = int(os.getenv('FLAT_MAX_CONTEXT_CHARS', '1800'))


@dataclass
class PromptConfig:
    answer_style: str = os.getenv('ANSWER_STYLE', 'concise, accurate, and cite supporting evidence')


@dataclass
class AppConfig:
    neo4j: Neo4jConfig = field(default_factory=Neo4jConfig)
    milvus: MilvusConfig = field(default_factory=MilvusConfig)
    jina: JinaConfig = field(default_factory=JinaConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    data: DataConfig = field(default_factory=DataConfig)
    build: BuildConfig = field(default_factory=BuildConfig)
    retrieval: RetrievalConfig = field(default_factory=RetrievalConfig)
    prompt: PromptConfig = field(default_factory=PromptConfig)


config = AppConfig()
config


AppConfig(neo4j=Neo4jConfig(uri='bolt://localhost:7687', user='neo4j', password='password', database='neo4j', fulltext_index_name='entity_fulltext'), milvus=MilvusConfig(uri='http://localhost:19530', token='', collection_name='graph_rag_chunks', metric_type='COSINE', drop_existing_collection=False), jina=JinaConfig(api_key='jina_f31eeb77e68740fba55dd3576761061ezgNqHZ0AhOj0e9BSW0-9rvgfrwAT', embedding_model='jina-embeddings-v5-text-small', api_base_url='https://api.jina.ai/v1/embeddings', normalized=True, embedding_type='float', truncate=True), model=ModelConfig(chat_model='gpt-5.4-mini', temperature=0.0), data=DataConfig(document_paths=['data/wiki100k_samples/01_12.md'], chunk_size=1400, chunk_overlap=180, max_documents=1), build=BuildConfig(max_entities_per_chunk=8, max_relationships_per_chunk=10, extraction_cache_path='.cache_graph_extraction.json', build_manifest_path='.cache_build_manifest.json', skip_existing_build=True, rebuild_neo4j=False, truncate_existing_graph=False), retriev

In [4]:
def require_non_empty(value: str, label: str) -> str:
    if not value or value.startswith('YOUR_'):
        raise ValueError(f'{label} is not configured. Please set it in .env.')
    return value


def make_llm() -> ChatOpenAI:
    return ChatOpenAI(
        model=require_non_empty(config.model.chat_model, 'OPENAI_CHAT_MODEL'),
        temperature=config.model.temperature,
    )


llm = make_llm()

print('Chat model:', config.model.chat_model)
print('Jina embedding model:', config.jina.embedding_model)

Chat model: gpt-5.4-mini
Jina embedding model: jina-embeddings-v5-text-small


In [5]:
@dataclass
class DocumentChunk:
    chunk_id: str
    source_path: str
    doc_id: str
    chunk_index: int
    text: str
    metadata: Dict[str, Any] = field(default_factory=dict)


def load_text_from_path(path: str) -> List[Dict[str, Any]]:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f'Document not found: {path}')

    if p.suffix.lower() in {'.md', '.txt', '.markdown'}:
        with p.open('r', encoding='utf-8') as f:
            text = f.read()
        return [
            {
                'source_path': str(p),
                'doc_id': p.stem,
                'page': None,
                'text': text,
            }
        ]

    raise ValueError(f'Unsupported document type: {p.suffix}. Use .md or .txt files generated from the dataset.')


def chunk_text(text: str, chunk_size: int, chunk_overlap: int) -> List[str]:
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []

    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(text):
            break
        start = max(0, end - chunk_overlap)
    return chunks


def build_chunks(document_paths: List[str]) -> List[DocumentChunk]:
    raw_docs: List[Dict[str, Any]] = []
    limit = config.data.max_documents
    selected_paths = document_paths if limit is None else document_paths[:limit]
    for path in selected_paths:
        raw_docs.extend(load_text_from_path(path))

    chunks: List[DocumentChunk] = []
    for doc in raw_docs:
        parts = chunk_text(doc['text'], config.data.chunk_size, config.data.chunk_overlap)
        for idx, part in enumerate(parts):
            fingerprint = hashlib.sha1(
                f"{doc['source_path']}:{doc['page']}:{idx}:{part}".encode('utf-8')
            ).hexdigest()
            chunks.append(
                DocumentChunk(
                    chunk_id=fingerprint,
                    source_path=doc['source_path'],
                    doc_id=doc['doc_id'],
                    chunk_index=idx,
                    text=part,
                    metadata={'page': doc['page']},
                )
            )
    return chunks


if not config.data.document_paths:
    raise ValueError('DOCUMENT_PATHS is not configured. Add at least one document path in .env.')

chunks = build_chunks(config.data.document_paths)
print(f'Loaded {len(chunks)} chunks from {len(config.data.document_paths)} document path(s).')
chunks[:2]

Loaded 38 chunks from 1 document path(s).


[DocumentChunk(chunk_id='3f528e2b8fb93e6d3e90aefb0fbbf8d32d20c606', source_path='data/wiki100k_samples/01_12.md', doc_id='01_12', chunk_index=0, text="--- id: 12 title: Anarchism url: https://en.wikipedia.org/wiki/Anarchism source: wiki100k --- # Anarchism Source: https://en.wikipedia.org/wiki/Anarchism Anarchism is a political philosophy and movement that is skeptical of all justifications for authority and seeks to abolish the institutions it claims maintain unnecessary coercion and hierarchy, typically including nation-states, and capitalism. Anarchism advocates for the replacement of the state with stateless societies and voluntary free associations. As a historically left-wing movement, this reading of anarchism is placed on the farthest left of the political spectrum, usually described as the libertarian wing of the socialist movement (libertarian socialism). Humans have lived in societies without formal hierarchies long before the establishment of states, realms, or empires. Wit

In [6]:
class JinaEmbeddingClient:
    def __init__(self):
        self.api_key = require_non_empty(config.jina.api_key, 'JINA_API_KEY')
        self.base_url = config.jina.api_base_url
        self.model = require_non_empty(config.jina.embedding_model, 'JINA_EMBEDDING_MODEL')
        self.normalized = config.jina.normalized
        self.embedding_type = config.jina.embedding_type
        self.truncate = config.jina.truncate
        self.session = requests.Session()
        self.session.headers.update(
            {
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {self.api_key}',
            }
        )

    def embed(self, texts: List[str]) -> List[List[float]]:
        if not texts:
            return []

        payload = {
            'model': self.model,
            'input': texts,
            'normalized': self.normalized,
            'embedding_type': self.embedding_type,
            'truncate': self.truncate,
        }
        response = self.session.post(self.base_url, json=payload, timeout=120)
        response.raise_for_status()
        data = response.json()

        items = data.get('data', data)
        vectors: List[List[float]] = []
        for item in items:
            if isinstance(item, dict) and 'embedding' in item:
                vectors.append(item['embedding'])
            else:
                vectors.append(item)
        return vectors


jina_client = JinaEmbeddingClient()


def embed_texts(texts: List[str], batch_size: int = 64) -> np.ndarray:
    vectors: List[List[float]] = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Embedding chunks'):
        batch = texts[i : i + batch_size]
        vectors.extend(jina_client.embed(batch))
    array = np.asarray(vectors, dtype=np.float32)
    norms = np.linalg.norm(array, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1.0, norms)
    return array / norms


class MilvusVectorIndex:
    def __init__(self):
        self.client = self._create_client()
        self.collection_name = config.milvus.collection_name
        self.dimension = len(jina_client.embed(['dimension probe'])[0])
        self.chunk_store: Dict[int, Dict[str, Any]] = {}
        self._collection_ready = False

    def _create_client(self) -> MilvusClient:
        require_non_empty(config.milvus.uri, 'MILVUS_URI')
        kwargs: Dict[str, Any] = {'uri': config.milvus.uri}
        if config.milvus.token:
            kwargs['token'] = config.milvus.token
        return MilvusClient(**kwargs)

    def prepare_collection(self) -> None:
        if self.client.has_collection(collection_name=self.collection_name):
            if config.milvus.drop_existing_collection:
                self.client.drop_collection(collection_name=self.collection_name)
        if not self.client.has_collection(collection_name=self.collection_name):
            self.client.create_collection(
                collection_name=self.collection_name,
                dimension=self.dimension,
                metric_type=config.milvus.metric_type,
            )
        self.client.load_collection(collection_name=self.collection_name)
        self._collection_ready = True

    def register_chunk_records(self, chunk_records: List[DocumentChunk]) -> None:
        for chunk in chunk_records:
            milvus_id = self._stable_int_id(chunk.chunk_id)
            self.chunk_store[milvus_id] = {
                'chunk_id': chunk.chunk_id,
                'text': chunk.text,
                'metadata': {
                    'source_path': chunk.source_path,
                    'doc_id': chunk.doc_id,
                    'chunk_index': chunk.chunk_index,
                    'page': chunk.metadata.get('page'),
                },
            }

    @staticmethod
    def _stable_int_id(chunk_id: str) -> int:
        return int(hashlib.sha1(chunk_id.encode('utf-8')).hexdigest()[:15], 16)

    def add(self, chunk_records: List[DocumentChunk], embeddings: Optional[np.ndarray] = None) -> None:
        if not chunk_records:
            return
        if not self._collection_ready:
            self.prepare_collection()

        self.register_chunk_records(chunk_records)
        vectors = embeddings if embeddings is not None else embed_texts([chunk.text for chunk in chunk_records])
        rows = []
        for chunk, vector in zip(chunk_records, vectors):
            milvus_id = self._stable_int_id(chunk.chunk_id)
            rows.append({'id': milvus_id, 'vector': vector.tolist()})

        self.client.insert(collection_name=self.collection_name, data=rows)
        self.client.flush(collection_name=self.collection_name)

    def search(self, query: str, top_k: int) -> List[Dict[str, Any]]:
        if not self.chunk_store or not self.client.has_collection(collection_name=self.collection_name):
            return []

        query_vector = jina_client.embed([query])[0]
        results = self.client.search(
            collection_name=self.collection_name,
            data=[query_vector],
            limit=top_k,
        )
        hits = results[0] if results else []
        payload: List[Dict[str, Any]] = []
        for hit in hits:
            if isinstance(hit, dict):
                milvus_id = hit.get('id')
                score = hit.get('distance', hit.get('score', 0.0))
            else:
                milvus_id = getattr(hit, 'id', None)
                score = getattr(hit, 'distance', 0.0)
            record = self.chunk_store.get(milvus_id)
            if not record:
                continue
            payload.append(
                {
                    'id': milvus_id,
                    'chunk_id': record['chunk_id'],
                    'text': record['text'],
                    'metadata': record['metadata'],
                    'score': float(score),
                }
            )
        return payload


vector_index = MilvusVectorIndex()
vector_index.prepare_collection()


In [7]:
class EntityRecord(BaseModel):
    name: str = Field(description="Canonical entity name")
    entity_type: str = Field(default="Unknown", description="Entity category")
    description: str = Field(default="", description="Short entity description grounded in the chunk")


class RelationshipRecord(BaseModel):
    source: str = Field(description="Source entity name")
    relation: str = Field(description="Relationship phrase, verb, or predicate")
    target: str = Field(description="Target entity name")
    evidence: str = Field(default="", description="Short evidence string from the chunk")


class ChunkExtraction(BaseModel):
    summary: str = Field(description="One-sentence summary of the chunk")
    entities: List[EntityRecord] = Field(default_factory=list)
    relationships: List[RelationshipRecord] = Field(default_factory=list)


class QuerySignals(BaseModel):
    intent: str = Field(description="Short description of the user intent")
    entities: List[str] = Field(default_factory=list)
    keywords: List[str] = Field(default_factory=list)


class MCQAnswer(BaseModel):
    answer: str = Field(description="Chosen option key such as A, B, C, or D; or 'I don't know' if the context is insufficient")
    reason: str = Field(description="Short reason grounded only in the provided context")


extraction_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You extract grounded knowledge from text. Return only information explicitly supported by the input chunk.",
        ),
        (
            "human",
            '''
            Extract entities and relationships from the following chunk.

            Rules:
            - Keep entity names canonical and concise.
            - Prefer real named entities, concepts, organizations, places, events, dates, and technical terms.
            - Relationships must be direct and grounded.
            - Do not invent facts.

            Chunk:
            {chunk_text}
            ''',
        ),
    ]
)

query_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You extract retrieval signals for GraphRAG. Return concise structured output.",
        ),
        (
            "human",
            '''
            Analyze the question and return:
            - intent: the core information need
            - entities: explicit or implied entities that should be used as graph seeds
            - keywords: important search terms

            Question:
            {question}
            ''',
        ),
    ]
)

answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You answer multiple-choice questions using only the provided context and the listed options. Do not use any outside knowledge, memory, or inference beyond what the context explicitly supports. Select exactly one option key from the options list, or respond with exactly 'I don't know.' for English questions or exactly 'Không biết.' for Vietnamese questions when the context is insufficient. Do not guess, do not hallucinate, and do not add facts that are not directly supported by the context.",
        ),
        (
            "human",
            '''
            Choose the single best option using only the Context section below.
            Return structured output with:
            - answer: exactly one option key (A, B, C, D, ...) or the exact fallback phrase if the context is insufficient
            - reason: a short, direct explanation grounded only in the context
            - Keep the reason concise.

            Question:
            {question}

            Options:
            {options}

            Context:
            {context}

            ''',
        ),
    ]
)

structured_extractor = llm.with_structured_output(ChunkExtraction)
structured_query_extractor = llm.with_structured_output(QuerySignals)
structured_answer = llm.with_structured_output(MCQAnswer)
answer_chain = answer_prompt | structured_answer

In [8]:
def get_extraction_cache(path: str) -> Dict[str, Any]:
    cache_path = Path(path)
    if cache_path.exists():
        return json.loads(cache_path.read_text(encoding="utf-8"))
    return {}


def save_extraction_cache(path: str, payload: Dict[str, Any]) -> None:
    Path(path).write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


extraction_cache = get_extraction_cache(config.build.extraction_cache_path)


def chunk_cache_key(chunk: DocumentChunk) -> str:
    return chunk.chunk_id


def extract_chunk(chunk: DocumentChunk) -> ChunkExtraction:
    cache_key = chunk_cache_key(chunk)
    if cache_key in extraction_cache:
        return ChunkExtraction.model_validate(extraction_cache[cache_key])

    result: ChunkExtraction = structured_extractor.invoke(
        extraction_prompt.format_messages(chunk_text=chunk.text)
    )
    extraction_cache[cache_key] = result.model_dump()
    return result


def extract_query_signals(question: str) -> QuerySignals:
    return structured_query_extractor.invoke(
        query_prompt.format_messages(question=question)
    )

In [9]:
def get_neo4j_driver() -> GraphDatabase.driver:
    require_non_empty(config.neo4j.uri, "NEO4J_URI")
    require_non_empty(config.neo4j.user, "NEO4J_USERNAME / NEO4J_USER")
    require_non_empty(config.neo4j.password, "NEO4J_PASSWORD")
    return GraphDatabase.driver(
        config.neo4j.uri,
        auth=(config.neo4j.user, config.neo4j.password),
    )


driver = get_neo4j_driver()


def neo4j_run(query: str, parameters: Optional[Dict[str, Any]] = None) -> List[Dict[str, Any]]:
    with driver.session(database=config.neo4j.database) as session:
        result = session.run(query, parameters or {})
        return [record.data() for record in result]


def ensure_neo4j_schema() -> None:
    statements = [
        '''
        CREATE CONSTRAINT entity_name_unique IF NOT EXISTS
        FOR (e:Entity) REQUIRE e.name IS UNIQUE
        ''',
        '''
        CREATE CONSTRAINT chunk_id_unique IF NOT EXISTS
        FOR (c:Chunk) REQUIRE c.chunk_id IS UNIQUE
        ''',
        '''
        CREATE FULLTEXT INDEX entity_fulltext IF NOT EXISTS
        FOR (e:Entity) ON EACH [e.name, e.description]
        ''',
    ]
    for statement in statements:
        neo4j_run(statement)


def truncate_graph() -> None:
    neo4j_run("MATCH (n) DETACH DELETE n")


def upsert_chunk(chunk: DocumentChunk, extraction: ChunkExtraction, embedding: List[float]) -> None:
    neo4j_run(
        '''
        MERGE (c:Chunk {chunk_id: $chunk_id})
        SET c.source_path = $source_path,
            c.doc_id = $doc_id,
            c.chunk_index = $chunk_index,
            c.text = $text,
            c.page = $page,
            c.embedding = $embedding
        ''',
        {
            "chunk_id": chunk.chunk_id,
            "source_path": chunk.source_path,
            "doc_id": chunk.doc_id,
            "chunk_index": chunk.chunk_index,
            "text": chunk.text,
            "page": chunk.metadata.get("page"),
            "embedding": embedding,
        },
    )

    neo4j_run(
        '''
        MATCH (c:Chunk {chunk_id: $chunk_id})
        SET c.summary = $summary
        ''',
        {"chunk_id": chunk.chunk_id, "summary": extraction.summary},
    )

    for entity in extraction.entities[: config.build.max_entities_per_chunk]:
        neo4j_run(
            '''
            MERGE (e:Entity {name: $name})
            SET e.entity_type = $entity_type,
                e.description = CASE
                    WHEN coalesce(e.description, '') = '' THEN $description
                    ELSE e.description
                END
            WITH e
            MATCH (c:Chunk {chunk_id: $chunk_id})
            MERGE (e)-[:MENTIONED_IN]->(c)
            ''',
            {
                "name": entity.name,
                "entity_type": entity.entity_type,
                "description": entity.description,
                "chunk_id": chunk.chunk_id,
            },
        )

    for rel in extraction.relationships[: config.build.max_relationships_per_chunk]:
        neo4j_run(
            '''
            MERGE (source:Entity {name: $source})
            MERGE (target:Entity {name: $target})
            MERGE (source)-[r:RELATED_TO {relation: $relation}]->(target)
            SET r.evidence = CASE
                WHEN coalesce(r.evidence, '') = '' THEN $evidence
                ELSE r.evidence
            END
            ''',
            {
                "source": rel.source,
                "target": rel.target,
                "relation": rel.relation,
                "evidence": rel.evidence,
            },
        )


ensure_neo4j_schema()
if config.build.truncate_existing_graph:
    truncate_graph()
print("Neo4j is ready.")

Neo4j is ready.


In [10]:


def load_build_manifest(path: str) -> Dict[str, Any]:
    manifest_path = Path(path)
    if manifest_path.exists():
        return json.loads(manifest_path.read_text(encoding='utf-8'))
    return {}


def save_build_manifest(path: str, payload: Dict[str, Any]) -> None:
    Path(path).write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')


def build_signature() -> str:
    payload = {
        'documents': config.data.document_paths,
        'chunk_size': config.data.chunk_size,
        'chunk_overlap': config.data.chunk_overlap,
        'max_documents': config.data.max_documents,
        'jina_model': config.jina.embedding_model,
        'milvus_collection': config.milvus.collection_name,
        'neo4j_database': config.neo4j.database,
    }
    return hashlib.sha1(json.dumps(payload, sort_keys=True).encode('utf-8')).hexdigest()


def get_neo4j_chunk_count() -> int:
    result = neo4j_run('MATCH (c:Chunk) RETURN count(c) AS count')
    return int(result[0]['count']) if result else 0


def get_milvus_chunk_count(expected_count: int) -> int:
    if not vector_index.client.has_collection(collection_name=vector_index.collection_name):
        return 0
    for method_name in ('get_collection_stats', 'describe_collection'):
        method = getattr(vector_index.client, method_name, None)
        if method is None:
            continue
        try:
            stats = method(collection_name=vector_index.collection_name)
        except TypeError:
            try:
                stats = method(vector_index.collection_name)
            except Exception:
                stats = None
        except Exception:
            stats = None
        if isinstance(stats, dict):
            for key in ('row_count', 'rowCount', 'num_entities', 'count'):
                if stats.get(key) is not None:
                    return int(stats[key])
        if hasattr(stats, 'get'):
            for key in ('row_count', 'rowCount', 'num_entities', 'count'):
                value = stats.get(key)
                if value is not None:
                    return int(value)
    try:
        rows = vector_index.client.query(
            collection_name=vector_index.collection_name,
            filter='id >= 0',
            output_fields=['id'],
            limit=expected_count,
        )
        return len(rows)
    except Exception:
        return 0


def drop_milvus_collection_if_needed(expected_chunks: int) -> None:
    if not vector_index.client.has_collection(collection_name=vector_index.collection_name):
        return
    current_count = get_milvus_chunk_count(expected_chunks)
    if current_count not in (0, expected_chunks):
        print(f'Dropping stale Milvus collection {vector_index.collection_name} with {current_count} rows.')
        vector_index.client.drop_collection(collection_name=vector_index.collection_name)
        vector_index._collection_ready = False


def print_build_status() -> None:
    expected_chunks = len(chunks)
    signature = build_signature()
    manifest = load_build_manifest(config.build.build_manifest_path)
    neo4j_count = get_neo4j_chunk_count()
    milvus_count = get_milvus_chunk_count(expected_chunks)
    reuse_possible = config.build.skip_existing_build and neo4j_count == expected_chunks and milvus_count == expected_chunks

    print('Build status:')
    print(f"- expected_chunks: {expected_chunks}")
    print(f"- neo4j_chunks: {neo4j_count}")
    print(f"- milvus_chunks: {milvus_count}")
    print(f"- skip_existing_build: {config.build.skip_existing_build}")
    print(f"- manifest_exists: {bool(manifest)}")
    print(f"- manifest_signature_match: {manifest.get('signature') == signature if manifest else False}")
    print(f"- reuse_possible: {reuse_possible}")


def build_graph_and_index() -> pd.DataFrame:
    expected_chunks = len(chunks)
    signature = build_signature()
    manifest = load_build_manifest(config.build.build_manifest_path)
    neo4j_count = get_neo4j_chunk_count()
    milvus_count = get_milvus_chunk_count(expected_chunks)
    if config.milvus.drop_existing_collection and milvus_count not in (0, expected_chunks):
        drop_milvus_collection_if_needed(expected_chunks)
        milvus_count = 0

    if config.build.skip_existing_build and neo4j_count == expected_chunks and milvus_count == expected_chunks:
        vector_index.prepare_collection()
        vector_index.register_chunk_records(chunks)
        print('Existing build detected; skipping rebuild.')
        rows = manifest.get('rows') if manifest.get('signature') == signature else None
        if rows:
            return pd.DataFrame(rows)
        cached_rows = []
        for chunk in chunks:
            cached = extraction_cache.get(chunk.chunk_id, {})
            cached_rows.append(
                {
                    'chunk_id': chunk.chunk_id,
                    'source_path': chunk.source_path,
                    'doc_id': chunk.doc_id,
                    'chunk_index': chunk.chunk_index,
                    'entities': [e.get('name') for e in cached.get('entities', []) if e.get('name')],
                    'relationships': [
                        f"{r.get('source')} -[{r.get('relation')}]-> {r.get('target')}"
                        for r in cached.get('relationships', [])
                        if r.get('source') and r.get('relation') and r.get('target')
                    ],
                    'summary': cached.get('summary', ''),
                }
            )
        return pd.DataFrame(cached_rows)

    rows = []
    chunk_texts = [chunk.text for chunk in chunks]
    chunk_embeddings = embed_texts(chunk_texts)
    vector_index.add(chunks, embeddings=chunk_embeddings)

    for chunk, embedding in tqdm(list(zip(chunks, chunk_embeddings)), desc='Building graph'):
        extraction = extract_chunk(chunk)
        upsert_chunk(chunk, extraction, embedding.tolist())
        rows.append(
            {
                'chunk_id': chunk.chunk_id,
                'source_path': chunk.source_path,
                'doc_id': chunk.doc_id,
                'chunk_index': chunk.chunk_index,
                'entities': [e.name for e in extraction.entities],
                'relationships': [
                    f'{r.source} -[{r.relation}]-> {r.target}' for r in extraction.relationships
                ],
                'summary': extraction.summary,
            }
        )

    save_extraction_cache(config.build.extraction_cache_path, extraction_cache)
    save_build_manifest(config.build.build_manifest_path, {'signature': signature, 'rows': rows, 'chunk_count': expected_chunks})
    return pd.DataFrame(rows)
print_build_status()
index_df = build_graph_and_index()
index_df.head()


Build status:
- expected_chunks: 38
- neo4j_chunks: 38
- milvus_chunks: 38
- skip_existing_build: True
- manifest_exists: True
- manifest_signature_match: False
- reuse_possible: True
Existing build detected; skipping rebuild.


,chunk_id,source_path,doc_id,chunk_index,entities,relationships,summary
0,3f528e2b8fb93e6d3e90aefb0fbbf8d32d20c606,data/wiki100k_samples/01_12.md,01_12,0,"[Anarchism, Authority, Nation-states, Capitalism, State, Stateless society, Voluntary free association, Libertarian socialism, Socialist movement, Enlightenment, 19th century, 20th century, Worker...","[Anarchism -[is a]-> Political philosophy, Anarchism -[is a]-> Movement, Anarchism -[is skeptical of]-> Authority, Anarchism -[seeks to abolish]-> Nation-states, Anarchism -[seeks to abolish]-> Ca...","Anarchism is described as a political philosophy and movement that opposes unjustified authority and seeks stateless societies, with modern anarchism emerging from the Enlightenment and flourishin..."
1,436df2a7113186ad55852390b20bc750df794c11,data/wiki100k_samples/01_12.md,01_12,1,"[anarchism, anarchist movement, Paris Commune, Russian Civil War, Spanish Civil War, classical era of anarchism, anti-capitalist movement, anti-war movement, anti-globalisation movement, revolutio...","[anarchists -[took part in]-> Paris Commune, anarchists -[took part in]-> Russian Civil War, anarchists -[took part in]-> Spanish Civil War, Spanish Civil War -[marked the end of]-> classical era ...","The chunk describes anarchism’s historical participation in revolutions, its later resurgence in anti-capitalist and anti-war movements, its revolutionary and evolutionary strategies, and the etym..."
2,ed3c7f8c4c415b820f256cdfcb494ff1816010b9,data/wiki100k_samples/01_12.md,01_12,2,"[anarchism, anarchy, anarchisme, French Revolution, William Godwin, Wilhelm Weitling, Pierre-Joseph Proudhon, libertarianism, free-market anarchism, libertarian anarchism, New Left, libertarian Ma...","[anarchism -[appears in English as]-> anarchisme, anarchy -[appears in English from]-> 1539, early English usages of anarchy -[emphasised]-> disorder, French Revolution -[factions labelled opponen...","The chunk describes the historical emergence and usage of the terms anarchism, anarchy, and libertarianism, and identifies key figures associated with early anarchist doctrine and the formal namin..."
3,0e33d72f7d950e76b575a2ed6ead8041ace86294,data/wiki100k_samples/01_12.md,01_12,3,"[New Left, libertarian Marxists, authoritarian socialists, vanguard party, extreme cultural liberals, civil liberties, anarchists, libertarian socialist, anarchism, socialist movement, socialist f...","[New Left -[includes]-> disparate groups, libertarian Marxists -[includes]-> disparate groups, New Left -[does not associate with]-> authoritarian socialists, libertarian Marxists -[does not assoc...","The chunk discusses anarchism as the anti-authoritarian wing of socialism, its relation to libertarian socialism and liberalism, and scholarly debates over its definition and misuse of anarcho-cap..."
4,2c018230f3b069d347ba39c711d2d7b82ca9fc85,data/wiki100k_samples/01_12.md,01_12,4,"[anarchism, state apparatus, non-coercive society, human nature, anarchy, China, Greece, Taoism, Zhuang Zhou, Laozi, Stoicism, Aeschylus, Sophocles, Antigone, Socrates, Athenian authorities, Cynic...","[anarchism -[rejects]-> state apparatus, anarchism -[envisions]-> non-coercive society, human nature -[allows]-> non-coercive society, anarchism -[pursues]-> anarchy, philosophical anarchism -[del...","The chunk describes anarchism as a non-coercive ideal and outlines early precursors in China, Greece, and medieval Europe."


In [11]:
def search_flat_chunks(question: str, top_k: Optional[int] = None) -> List[Dict[str, Any]]:
    top_k = top_k or config.retrieval.flat_top_k
    return vector_index.search(question, top_k=top_k)


def format_options(options: Dict[str, str]) -> str:
    return '\n'.join(f"{key}. {options[key]}" for key in sorted(options.keys()))


def normalize_mcq_answer(raw_answer: str, allowed_keys: List[str]) -> str:
    normalized = (raw_answer or '').strip()
    upper = normalized.upper()
    fallback_markers = ["I DON'T KNOW", 'I DONT KNOW', 'KHÔNG BIẾT', 'KHONG BIET']
    if any(marker in upper for marker in fallback_markers):
        return "I don't know" if "I DON'T KNOW" in upper or "I DONT KNOW" in upper else 'Không biết'

    for key in sorted({key.upper() for key in allowed_keys}, key=len, reverse=True):
        if re.search(rf'(?<![A-Z0-9]){re.escape(key)}(?![A-Z0-9])', upper):
            return key

    return normalized


def invoke_mcq_answer(question: str, options: Dict[str, str], context: str) -> Dict[str, Any]:
    result: MCQAnswer = answer_chain.invoke(
        {
            'question': question,
            'options': format_options(options),
            'context': context,
        }
    )
    selected_option = normalize_mcq_answer(result.answer, list(options.keys()))
    reason = result.reason.strip()
    return {
        'selected_option': selected_option,
        'reason': reason,
        'raw': result.model_dump(),
    }


def format_choice_answer(selected_option: str, options: Dict[str, str], reason: str) -> str:
    choice_text = options.get(selected_option, '')
    if choice_text:
        answer_line = f"{selected_option}. {choice_text}"
    else:
        answer_line = selected_option
    if reason:
        return f"{answer_line}\nReason: {reason}"
    return answer_line


def format_chunks(chunks_payload: List[Dict[str, Any]]) -> str:
    lines = []
    for i, item in enumerate(chunks_payload, start=1):
        meta = item.get('metadata', {})
        lines.append(
            f"[Chunk {i}] score={item.get('score', 0):.4f} | source={meta.get('source_path')} | doc={meta.get('doc_id')} | page={meta.get('page')} | chunk_index={meta.get('chunk_index')}\n"
            f"{item.get('text', '')}"
        )
    return '\n\n'.join(lines)


def answer_flat_rag(question: str, options: Dict[str, str]) -> Dict[str, Any]:
    # Deliberately naive baseline: use only the single most relevant chunk.
    retrieved = search_flat_chunks(question, 1)[:1]
    context = format_chunks(retrieved)[: config.retrieval.flat_max_context_chars]
    mcq = invoke_mcq_answer(question, options, context)
    return {
        'mode': 'flat_rag',
        'question': question,
        'selected_option': mcq['selected_option'],
        'selected_text': options.get(mcq['selected_option'], ''),
        'reason': mcq['reason'],
        'answer': format_choice_answer(mcq['selected_option'], options, mcq['reason']),
        'retrieved_chunks': retrieved,
        'context': context,
        'raw_response': mcq['raw'],
    }


In [12]:
def lookup_seed_entities(question: str) -> List[Dict[str, Any]]:
    signals = extract_query_signals(question)
    candidates: List[Dict[str, Any]] = []
    for entity_name in signals.entities:
        rows = neo4j_run(
            '''
            CALL db.index.fulltext.queryNodes($index_name, $query) YIELD node, score
            RETURN node.name AS name, node.entity_type AS entity_type, node.description AS description, score
            ORDER BY score DESC
            LIMIT $limit
            ''',
            {
                'index_name': config.neo4j.fulltext_index_name,
                'query': entity_name,
                'limit': config.retrieval.graph_seed_k,
            },
        )
        candidates.extend(rows)

    if candidates:
        dedup = {}
        for row in candidates:
            dedup[row['name']] = row
        return list(dedup.values())

    fallback = search_flat_chunks(question, config.retrieval.graph_chunk_boost_k)
    fallback_entities: List[Dict[str, Any]] = []
    for item in fallback:
        chunk_id = item['id']
        rows = neo4j_run(
            '''
            MATCH (e:Entity)-[:MENTIONED_IN]->(c:Chunk {chunk_id: $chunk_id})
            RETURN DISTINCT e.name AS name, e.entity_type AS entity_type, e.description AS description
            LIMIT $limit
            ''',
            {'chunk_id': chunk_id, 'limit': config.retrieval.graph_seed_k},
        )
        fallback_entities.extend(rows)
    dedup = {}
    for row in fallback_entities:
        dedup[row['name']] = row
    return list(dedup.values())


def fetch_neighbors(entity_name: str, limit_per_node: int) -> List[Dict[str, Any]]:
    return neo4j_run(
        '''
        MATCH (e:Entity {name: $name})-[r:RELATED_TO]-(t:Entity)
        RETURN DISTINCT
            startNode(r).name AS source,
            r.relation AS relation,
            endNode(r).name AS target,
            coalesce(r.evidence, '') AS evidence
        ORDER BY source, relation, target
        LIMIT $limit
        ''',
        {'name': entity_name, 'limit': limit_per_node},
    )


def expand_entity_graph(seed_entities: List[str], hops: int, limit_per_node: int) -> Tuple[List[Dict[str, Any]], List[str]]:
    seen = set(seed_entities)
    frontier = list(seed_entities)
    triples: List[Dict[str, Any]] = []
    related_entities: List[str] = list(seed_entities)

    for _ in range(hops):
        next_frontier: List[str] = []
        for entity in frontier:
            neighbors = fetch_neighbors(entity, limit_per_node)
            for row in neighbors:
                triples.append(row)
                source = row['source']
                target = row['target']
                if source not in seen:
                    seen.add(source)
                    next_frontier.append(source)
                    related_entities.append(source)
                if target not in seen:
                    seen.add(target)
                    next_frontier.append(target)
                    related_entities.append(target)
        frontier = next_frontier
        if not frontier:
            break

    unique_triples = []
    dedup = set()
    for row in triples:
        key = (row['source'], row['relation'], row['target'], row.get('evidence', ''))
        if key not in dedup:
            dedup.add(key)
            unique_triples.append(row)
    return unique_triples, sorted(set(related_entities))


def fetch_supporting_chunks(entity_names: List[str], limit_per_entity: int) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    for entity_name in entity_names:
        rows.extend(
            neo4j_run(
                '''
                MATCH (e:Entity {name: $name})-[:MENTIONED_IN]->(c:Chunk)
                WITH c, properties(c) AS props
                RETURN DISTINCT c.chunk_id AS chunk_id, c.text AS text, c.source_path AS source_path,
                                c.doc_id AS doc_id, c.chunk_index AS chunk_index, props.page AS page
                ORDER BY c.chunk_index
                LIMIT $limit
                ''',
                {'name': entity_name, 'limit': limit_per_entity},
            )
        )
    dedup = {}
    for row in rows:
        dedup[row['chunk_id']] = row
    return list(dedup.values())


def format_triples(triples: List[Dict[str, Any]]) -> str:
    lines = []
    for i, row in enumerate(triples, start=1):
        evidence = row.get('evidence', '')
        if evidence:
            lines.append(
                f"[Triplet {i}] {row['source']} -[{row['relation']}]-> {row['target']} | evidence: {evidence}"
            )
        else:
            lines.append(f"[Triplet {i}] {row['source']} -[{row['relation']}]-> {row['target']}")
    return '\n'.join(lines)


def answer_graph_rag(question: str, options: Dict[str, str]) -> Dict[str, Any]:
    signals = extract_query_signals(question)
    seed_rows = lookup_seed_entities(question)
    seed_names = [row['name'] for row in seed_rows]

    if not seed_names:
        seed_names = signals.entities[: config.retrieval.graph_seed_k]

    triples, related_entities = expand_entity_graph(
        seed_entities=seed_names,
        hops=config.retrieval.graph_hops,
        limit_per_node=config.retrieval.graph_neighbor_limit_per_node,
    )

    supporting_chunks = fetch_supporting_chunks(
        entity_names=related_entities,
        limit_per_entity=config.retrieval.graph_chunk_boost_k,
    )

    vector_chunks = search_flat_chunks(question, config.retrieval.graph_chunk_boost_k)
    merged_chunks: List[Dict[str, Any]] = []
    dedup = set()
    for row in supporting_chunks + vector_chunks:
        chunk_id = row['chunk_id'] if 'chunk_id' in row else row.get('id')
        if chunk_id in dedup:
            continue
        dedup.add(chunk_id)
        if 'chunk_id' not in row:
            row = {
                'chunk_id': row['id'],
                'text': row['text'],
                'source_path': row['metadata'].get('source_path'),
                'doc_id': row['metadata'].get('doc_id'),
                'chunk_index': row['metadata'].get('chunk_index'),
                'page': row['metadata'].get('page'),
            }
        merged_chunks.append(row)

    merged_chunks = merged_chunks[: config.retrieval.max_context_chunks]

    graph_context = format_triples(triples)
    chunk_context = '\n\n'.join(
        [
            f"[Chunk {i}] source={row.get('source_path')} | doc={row.get('doc_id')} | page={row.get('page')} | chunk_index={row.get('chunk_index')}\n{row.get('text', '')}"
            for i, row in enumerate(merged_chunks, start=1)
        ]
    )
    context = f"GRAPH FACTS\n{graph_context}\n\nRETRIEVED CHUNKS\n{chunk_context}"
    context = context[: config.retrieval.max_context_chars]

    mcq = invoke_mcq_answer(question, options, context)

    return {
        'mode': 'graph_rag',
        'question': question,
        'selected_option': mcq['selected_option'],
        'selected_text': options.get(mcq['selected_option'], ''),
        'reason': mcq['reason'],
        'answer': format_choice_answer(mcq['selected_option'], options, mcq['reason']),
        'seed_entities': seed_names,
        'signals': signals.model_dump(),
        'triples': triples,
        'related_entities': related_entities,
        'supporting_chunks': merged_chunks,
        'context': context,
        'raw_response': mcq['raw'],
    }


In [13]:
def compare_flat_and_graph(case: Dict[str, Any]) -> pd.DataFrame:
    question = case['question']
    options = case['options']
    expected_answer = case['answer']
    flat_result = answer_flat_rag(question, options)
    graph_result = answer_graph_rag(question, options)

    rows = [
        {
            'approach': 'FlatRAG',
            'selected_option': flat_result['selected_option'],
            'selected_text': flat_result['selected_text'],
            'reason': flat_result['reason'],
            'answer': flat_result['answer'],
            'retrieved_items': len(flat_result['retrieved_chunks']),
            'evidence_type': 'semantic chunks only',
            'correct': flat_result['selected_option'] == expected_answer,
        },
        {
            'approach': 'GraphRAG',
            'selected_option': graph_result['selected_option'],
            'selected_text': graph_result['selected_text'],
            'reason': graph_result['reason'],
            'answer': graph_result['answer'],
            'retrieved_items': len(graph_result['supporting_chunks']) + len(graph_result['triples']),
            'evidence_type': 'graph triples + chunks',
            'correct': graph_result['selected_option'] == expected_answer,
        },
    ]
    return pd.DataFrame(rows)


def load_test_cases(path: str = 'test_data.json') -> List[Dict[str, Any]]:
    payload = json.loads(Path(path).read_text(encoding='utf-8'))
    cases = payload.get('cases', payload) if isinstance(payload, dict) else payload
    if not isinstance(cases, list):
        raise ValueError('test_data.json must contain a list of cases or a {"cases": [...]} object.')
    return cases


def normalize_answer_key(value: str) -> str:
    return re.sub(r'[^A-Z0-9]+', '', (value or '').upper().strip())


def evaluate_case(case: Dict[str, Any]) -> Dict[str, Any]:
    question = case['question']
    options = case['options']
    expected_answer = case['answer']
    flat_result = answer_flat_rag(question, options)
    graph_result = answer_graph_rag(question, options)

    flat_correct = normalize_answer_key(flat_result['selected_option']) == normalize_answer_key(expected_answer)
    graph_correct = normalize_answer_key(graph_result['selected_option']) == normalize_answer_key(expected_answer)

    if flat_correct and not graph_correct:
        preferred = 'FlatRAG'
    elif graph_correct and not flat_correct:
        preferred = 'GraphRAG'
    else:
        preferred = 'Tie'

    return {
        'id': case.get('id'),
        'language': case.get('language'),
        'question_type': case.get('question_type'),
        'difficulty': case.get('difficulty'),
        'question': question,
        'options': options,
        'expected_answer': expected_answer,
        'expected_answer_text': options.get(expected_answer, ''),
        'expected_focus': case.get('expected_focus', ''),
        'expected_system_preference': case.get('expected_system_preference', ''),
        'flat_selected_option': flat_result['selected_option'],
        'flat_selected_text': flat_result['selected_text'],
        'flat_reason': flat_result['reason'],
        'flat_correct': flat_correct,
        'graph_selected_option': graph_result['selected_option'],
        'graph_selected_text': graph_result['selected_text'],
        'graph_reason': graph_result['reason'],
        'graph_correct': graph_correct,
        'graph_evidence_items': len(graph_result['supporting_chunks']) + len(graph_result['triples']),
        'flat_evidence_items': len(flat_result['retrieved_chunks']),
        'preferred_system': preferred,
        'flat_answer_preview': flat_result['answer'][:300],
        'graph_answer_preview': graph_result['answer'][:300],
        'flat_answer': flat_result['answer'],
        'graph_answer': graph_result['answer'],
        'flat_retrieved_chunks': flat_result['retrieved_chunks'],
        'graph_supporting_chunks': graph_result['supporting_chunks'],
        'graph_triples': graph_result['triples'],
    }


test_cases = load_test_cases()
evaluation_results = [evaluate_case(case) for case in test_cases]
evaluation_df = pd.DataFrame(
    [
        {
            'id': row['id'],
            'language': row['language'],
            'question_type': row['question_type'],
            'difficulty': row['difficulty'],
            'question': row['question'],
            'expected_answer': row['expected_answer'],
            'expected_answer_text': row['expected_answer_text'],
            'flat_selected_option': row['flat_selected_option'],
            'flat_selected_text': row['flat_selected_text'],
            'flat_correct': row['flat_correct'],
            'graph_selected_option': row['graph_selected_option'],
            'graph_selected_text': row['graph_selected_text'],
            'graph_correct': row['graph_correct'],
            'flat_evidence_items': row['flat_evidence_items'],
            'graph_evidence_items': row['graph_evidence_items'],
            'preferred_system': row['preferred_system'],
            'expected_system_preference': row['expected_system_preference'],
            'flat_reason_preview': row['flat_reason'][:180],
            'graph_reason_preview': row['graph_reason'][:180],
        }
        for row in evaluation_results
    ]
)

evaluation_summary = pd.DataFrame(
    {
        'metric': [
            'cases',
            'flat_correct',
            'graph_correct',
            'flat_accuracy',
            'graph_accuracy',
            'flat_only_wins',
            'graph_only_wins',
            'ties',
            'both_correct',
            'both_wrong',
        ],
        'value': [
            len(evaluation_results),
            int(evaluation_df['flat_correct'].sum()),
            int(evaluation_df['graph_correct'].sum()),
            float(evaluation_df['flat_correct'].mean()),
            float(evaluation_df['graph_correct'].mean()),
            int(((evaluation_df['flat_correct'] == True) & (evaluation_df['graph_correct'] == False)).sum()),
            int(((evaluation_df['graph_correct'] == True) & (evaluation_df['flat_correct'] == False)).sum()),
            int((evaluation_df['preferred_system'] == 'Tie').sum()),
            int(((evaluation_df['flat_correct'] == True) & (evaluation_df['graph_correct'] == True)).sum()),
            int(((evaluation_df['flat_correct'] == False) & (evaluation_df['graph_correct'] == False)).sum()),
        ],
    }
)

evaluation_summary


,metric,value
0,cases,20.00
1,flat_correct,17.00
2,graph_correct,19.00
3,flat_accuracy,0.85
4,graph_accuracy,0.95
5,flat_only_wins,1.00
6,graph_only_wins,3.00
7,ties,16.00
8,both_correct,16.00
9,both_wrong,0.00


In [18]:
# Detailed comparison for the first test case.
for i in range(0, len(evaluation_results)):
    case = evaluation_results[i]
    print(f"Case: {case['id']} | language={case['language']} | type={case['question_type']}")
    print(f"Question: {case['question']}")
    print(f"Expected answer: {case['expected_answer']} -> {case['expected_answer_text']}")
    print(f"Expected system preference: {case.get('expected_system_preference', '')}")
    print('Options:')
    for key in sorted(case['options'].keys()):
        print(f"  {key}. {case['options'][key]}")
    print()
    print('FlatRAG answer:')
    print(case['flat_answer'])
    print(f"Flat correct: {case['flat_correct']}")
    print()
    print('GraphRAG answer:')
    print(case['graph_answer'])
    print(f"Graph correct: {case['graph_correct']}")
    print()
    print(f"Flat evidence items: {case['flat_evidence_items']}")
    print(f"Graph evidence items: {case['graph_evidence_items']}")
    print(f"Heuristic preferred system: {case['preferred_system']}")
    print('-' * 80)


Case: None | language=None | type=None
Question: Một nhóm trong một liên minh lao động quốc tế đã bị loại bỏ sau xung đột tư tưởng, rồi lập ra một tổ chức riêng. Nhóm này gắn với cá nhân nào?
Expected answer: B -> Mikhail Bakunin
Expected system preference: 
Options:
  A. Karl Marx
  B. Mikhail Bakunin
  C. Pierre-Joseph Proudhon
  D. Peter Kropotkin

FlatRAG answer:
B. Mikhail Bakunin
Reason: The context says Bakuninists were expelled from the International after disputes and then responded to their expulsion by forming a new organization.
Flat correct: True

GraphRAG answer:
B. Mikhail Bakunin
Reason: Context says Bakunin’s faction was expelled from the International by the Marxists after disputes, and they then formed their own organization.
Graph correct: True

Flat evidence items: 1
Graph evidence items: 5
Heuristic preferred system: Tie
--------------------------------------------------------------------------------
Case: None | language=None | type=None
Question: Một biến cố vừa